# 06 · Hypothesentest — Favorita

## 0 · Imports & Setup

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
sys.path.append(os.path.abspath('../03_src'))

from utilis import mape, rmse, run_sarimax, run_prophet, run_xgb, run_lgbm, to_nf_format, compute_metrics, load_preds
from config import FINAL, TARGET_COL, FEATURE_COLS, EXOG_COLS, STAT_EXOG, HIST_EXOG, TRAIN_END, VAL_END, LOOKBACK, HORIZON, MODELL_ORDER, MODELL_COLORS, PRED_COLS, RESULTS, FILE_NAMES


## 1 - Daten laden

In [13]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train     = df.filter(pl.col('date') <= TRAIN_END)
val       = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test      = df.filter(pl.col('date') > VAL_END)
train_val = df.filter(pl.col('date') <= VAL_END)

stores = sorted(df['store_nbr'].unique().to_list())

print(f'Train:     {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:       {val.shape}    {val["date"].min()} → {val["date"].max()}')
print(f'Test:      {test.shape}   {test["date"].min()} → {test["date"].max()}')
print(f'Stores:    {len(stores)}')

Train:     (70114, 37)  2013-01-29 → 2016-12-31
Val:       (7965, 37)    2017-01-01 → 2017-05-31
Test:      (4104, 37)   2017-06-01 → 2017-08-15
Stores:    54


In [12]:
test

<module 'pandas' from 'c:\\Users\\maxkr\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\pandas\\__init__.py'>

## 2 · Ausgangsdatensatz

In [9]:
models_cfg = {
    'XGBoost':  'pred_xgb',
    'LightGBM': 'pred_lgbm',
    'PatchTST': 'pred_patchtst',
    'NHITS':    'pred_nhits',
    'SARIMAX':  'pred_sarimax',
    'Prophet':  'pred_prophet',
}
models_cfg

{'XGBoost': 'pred_xgb',
 'LightGBM': 'pred_lgbm',
 'PatchTST': 'pred_patchtst',
 'NHITS': 'pred_nhits',
 'SARIMAX': 'pred_sarimax',
 'Prophet': 'pred_prophet'}

In [11]:
base = (
    test.select(['store_nbr', 'date', TARGET_COL])
    .rename({TARGET_COL: 'y_true'})
    .to_pandas()
)


AttributeError: module 'pandas' has no attribute 'select'

In [ ]:

for name, col in models_cfg.items():
    fname = f'test_{name.lower()}.parquet'
    try:
        pred_df = pl.read_parquet(RESULTS / fname).to_pandas()
        base = base.merge(pred_df[['store_nbr', 'date', col]],
                          on=['store_nbr', 'date'], how='left')
    except FileNotFoundError:
        print(f'Nicht gefunden: {fname}')

# Store-Typ joinen
stores_pd = pl.read_parquet(FINAL / 'stores.parquet').to_pandas()
base = base.merge(stores_pd[['store_nbr', 'type', 'cluster']],
                  on='store_nbr', how='left')

# ── Metriken je Store ─────────────────────────────────────
store_metrics = []
for store_id, grp in base.groupby('store_nbr'):
    for model_name, col in models_cfg.items():
        if col not in grp.columns or grp[col].isna().all():
            continue
        mask   = grp[col].notna() & grp['y_true'].notna()
        y_true = grp.loc[mask, 'y_true'].values
        y_pred = grp.loc[mask, col].values
        store_metrics.append({
            'store_nbr':  store_id,
            'model':      model_name,
            'store_type': grp['type'].iloc[0],
            'MAE':        mean_absolute_error(y_true, y_pred),
            'RMSE':       rmse(y_true, y_pred),
            'MAPE':       mape(y_true, y_pred),
            'std_true':   y_true.std(),
            'acf_lag7':   pd.Series(y_true).autocorr(lag=7),
        })

metrics_df = pd.DataFrame(store_metrics)

# ── Gesamtübersicht ───────────────────────────────────────
summary = metrics_df.groupby('model')[['MAE', 'RMSE', 'MAPE']].mean().round(2)
print(summary.sort_values('MAE'))